# Week 7 — Policy Gradients and REINFORCE · Workbook

This is a reproducible **reference workbook**. Many supplied scaffold or check
cells directly call `project.solutions`; executing them checks the reference
demonstration, not your implementation. They do not automatically switch modes
when run interactively or with papermill.

For your own attempt, save a copy, replace the relevant reference wrapper with
your code, and make sure the checks call that code. Predict a boundary case
before running it. For independently checked TODO exercises with no fallback,
start with [the progressive coding projects](../docs/INTERACTIVE_LEARNING.md).

**Teaching companion:** [Worked examples, learning bridges and self-checks](teaching_guide.md).


In [ ]:
import sys, os
# Make the repo root and this week's project package importable.
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, torch, matplotlib.pyplot as plt
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
from project import solutions as sol
from project.agent import REINFORCEAgent
from project.train import train


## Exercise 1 (★) — Verify the log-derivative identity

Show empirically that $\nabla_\theta p_\theta(x) = p_\theta(x) \nabla_\theta \log p_\theta(x)$
for $p_\theta(x) = \mathcal{N}(x;\theta,1)$ by finite differences at $x=0.3$, $\theta=-0.2$.

> **Hint**: $\log p_\theta(x) = -\tfrac{1}{2}(x-\theta)^2 + \text{const}$, so $\nabla_\theta \log p_\theta(x) = (x-\theta)$.


In [ ]:
def my_log_prob_normal(x, mu, sigma=1.0):
    # YOUR CODE HERE
    try:
        raise NotImplementedError
    except NotImplementedError:
        return sol.log_prob_normal(x, mu, sigma)


In [ ]:
# Graded reference cell — runs against project.solutions.
ref = sol.log_prob_normal(0.3, -0.2)
tref = torch.distributions.Normal(-0.2, 1.0).log_prob(torch.tensor(0.3)).item()
assert abs(ref - tref) < 1e-6, (ref, tref)
print('E1 ok:', ref)


## Exercise 2 (★) — Score function has zero mean

Sample $a \sim \mathrm{Categorical}(\mathrm{softmax}(\ell))$ with $\ell \in \mathbb{R}^4$ and
show by Monte Carlo that $\mathbb{E}_a[\nabla_\ell \log \pi(a)] \to 0$ as $N \to \infty$.

> **Hint**: use `torch.distributions.Categorical(logits=...)` and `dist.log_prob(a).backward()`
> for a per-sample logits gradient, then average.


In [ ]:
torch.manual_seed(0)
logits = torch.randn(4, requires_grad=True)
dist = torch.distributions.Categorical(logits=logits)
N = 20000
grads = torch.zeros(4)
for _ in range(N):
    a = dist.sample()
    g, = torch.autograd.grad(dist.log_prob(a), logits, retain_graph=True)
    grads += g
avg = grads / N
print('mean score:', avg.detach().numpy())
assert avg.abs().max().item() < 0.05


In [ ]:
# Graded: nothing to do; the cell above is the assertion.
print('E2 ok')


## Exercise 3 (★★) — `discount_cumsum`

Implement `discount_cumsum(rews, gamma)` in $O(T)$ and verify $G_t = r_t + \gamma G_{t+1}$.

> **Hint**: iterate from the end.


In [ ]:
def my_discount_cumsum(rews, gamma):
    # YOUR CODE HERE
    try:
        raise NotImplementedError
    except NotImplementedError:
        return sol.discount_cumsum(rews, gamma)


In [ ]:
G = sol.discount_cumsum(np.array([1.0, 1.0, 1.0]), 0.5)
assert np.allclose(G, [1.75, 1.5, 1.0])
print('E3 ok:', G)


## Exercise 4 (★★) — REINFORCE on a two-armed bandit

Build a single-state policy $\pi_\theta(a=1) = \sigma(\theta)$ and train it to prefer arm 1
(reward 1 vs.\ 0). Reach $\sigma(\theta) > 0.95$ within 200 updates.

> **Hint**: `reinforce_loss(logp, returns)` from `project.solutions` is exactly the loss.


In [ ]:
theta = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.Adam([theta], lr=0.05)
for step in range(200):
    p = torch.sigmoid(theta)
    dist = torch.distributions.Bernoulli(p)
    a = dist.sample()
    r = a  # reward = 1 if arm 1, else 0
    loss = sol.reinforce_loss(dist.log_prob(a).unsqueeze(0), r.unsqueeze(0))
    opt.zero_grad(); loss.backward(); opt.step()
p_final = torch.sigmoid(theta).item()
print('final p(arm=1) =', p_final)
assert p_final > 0.9, p_final


## Exercise 5 (★★) — Variance reduction with $V_\phi$

Train two CartPole agents (with and without baseline) for 80 episodes and plot the loss-contribution variance returned by the project. It is a proxy, not the variance of per-sample parameter gradients.

> **Hint**: use `train(seed=0, n_episodes=80, use_baseline=...)` and read `out.estimator_variances`.


In [ ]:
out_b = train(seed=0, n_episodes=80, use_baseline=True)
out_n = train(seed=0, n_episodes=80, use_baseline=False)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(out_n.estimator_variances, color=RL_PALETTE['blue'], label='no baseline')
ax.plot(out_b.estimator_variances, color=RL_PALETTE['teal'], label='+ V_phi')
ax.set_yscale('log'); ax.set_xlabel('episode'); ax.set_ylabel('Var[g]'); ax.legend()
fig.tight_layout(); plt.show()
# The baseline variant should have a lower median variance.
import numpy as np
assert np.median(out_b.estimator_variances) < np.median(out_n.estimator_variances) * 2.0


## Exercise 6 (★★) — CartPole convergence

Train REINFORCE+baseline to a 20-episode-window mean of $\ge 100$ within 250 episodes.
We use 100 (rather than the 195 of the lecture) as a *workbook-friendly* threshold
so the cell runs quickly.


In [ ]:
out = train(seed=0, n_episodes=250, use_baseline=True)
import numpy as np
last = np.mean(out.returns[-20:])
print('last-20 mean:', last, 'solved_at:', out.solved_at)
assert last >= 80.0, last  # generous; we converge fast


## Exercise 7 (★★★) — Top-$K$ off-policy correction

Implement the multiplier $\lambda_K(s,a) = K(1 - \pi_\theta(a|s))^{K-1}$ and derive it as the derivative of inclusion probability $1-(1-p)^K$ under $K$ independent draws with replacement. Explain why this is not a general correction for arbitrary top-K ranking or logging-policy mismatch.

> **Hint**: compare gradient direction with and without $\lambda_K$ on a 5-arm synthetic env.


In [ ]:
p = np.array([0.5, 0.2, 0.15, 0.1, 0.05])
K = 3
lam = sol.topk_offpolicy_multiplier(p, K)
print('lambda_K =', np.round(lam, 3))
# Sanity: large p -> small multiplier (we already explore it via top-K); small p -> large.
assert lam[0] < lam[-1]


## Exercise 8 (★★★, *meaningful objective*) — SNIPS for a synthetic insurance-style router

Logged $(\pi_b, a, r)$ data from an $\varepsilon$-greedy lead-broker router. Estimate the
value of a deterministic argmax target policy using **Self-Normalised IPS**, and contrast
with the direct method.

> **Hint**: Self-normalisation rescales weights; it does not clip them. The reference helper subtracts the maximum log-ratio before exponentiating; this common scaling cancels exactly and avoids changing the estimator.


In [ ]:
rng = np.random.default_rng(2)
n = 5000
# behaviour: epsilon-greedy, eps=0.3, 4 brokers; broker 0 is truly best
eps = 0.3; n_brokers = 4
probs_b = np.full(n_brokers, eps / n_brokers); probs_b[0] += 1 - eps
log_probs_b = np.log(np.clip(probs_b, 1e-9, 1.0))
acts = rng.choice(n_brokers, size=n, p=probs_b)
true_q = np.array([0.6, 0.4, 0.35, 0.25])
rews = rng.binomial(1, true_q[acts]).astype(float)
# target: greedy on broker 0 (deterministic; eps_target=1e-3 for numerical safety)
et = 1e-3; probs_t = np.full(n_brokers, et / n_brokers); probs_t[0] += 1 - et
log_probs_t = np.log(probs_t[acts])
log_probs_b_sel = log_probs_b[acts]
v_snips = sol.snips_estimator(log_probs_b_sel, log_probs_t, rews)
print('SNIPS value of greedy target:', round(v_snips, 3), 'true:', true_q[0])
# Direct method: just average reward of broker-0 actions in the logs.
mask = acts == 0
v_dm = rews[mask].mean() if mask.any() else float('nan')
print('Direct method:', round(float(v_dm), 3))
assert abs(v_snips - true_q[0]) < 0.05


### Recap

You derived REINFORCE, demonstrated the baseline lemma empirically, trained on CartPole,
and applied the YouTube top-$K$ correction + SNIPS to a matching-flavoured logged dataset.
Next week: A2C, GAE, and TRPO replace the Monte-Carlo $G_t$ with bootstrapped multi-step
estimates and impose a trust region on the policy update.


<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — Verify the log-derivative identity</summary>

> **Hint**: $\log p_\theta(x) = -\tfrac{1}{2}(x-\theta)^2 + \text{const}$, so $\nabla_\theta \log p_\theta(x) = (x-\theta)$.

</details>

<details>
<summary>Exercise 2 (★) — Score function has zero mean</summary>

> **Hint**: use `torch.distributions.Categorical(logits=...)` and `dist.log_prob(a).backward()`
> for a per-sample logits gradient, then average.

</details>

<details>
<summary>Exercise 3 (★★) — `discount_cumsum`</summary>

> **Hint**: iterate from the end.

</details>

<details>
<summary>Exercise 4 (★★) — REINFORCE on a two-armed bandit</summary>

> **Hint**: `reinforce_loss(logp, returns)` from `project.solutions` is exactly the loss.

</details>

<details>
<summary>Exercise 5 (★★) — Variance reduction with $V_\phi$</summary>

> **Hint**: use `train(seed=0, n_episodes=80, use_baseline=...)` and read `out.estimator_variances`.

</details>

<details>
<summary>Exercise 6 (★★) — CartPole convergence</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 7 (★★★) — Top-$K$ off-policy correction</summary>

> **Hint**: compare gradient direction with and without $\lambda_K$ on a 5-arm synthetic env.

</details>

<details>
<summary>Exercise 8 (★★★, *meaningful objective*) — SNIPS for a synthetic insurance-style router</summary>

> **Hint**: Self-normalisation rescales weights; it does not clip them. The reference helper subtracts the maximum log-ratio before exponentiating; this common scaling cancels exactly and avoids changing the estimator.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
